In [ ]:
import itertools
import json
import sys
import gc
from pathlib import Path
import numpy as np
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

In [ ]:
EPOCHS = 1000
BATCH_SIZE = 64
ACT_FUNC = tf.nn.relu
MOMENTUM = 0.5

In [ ]:
norm_options = ['norm', 'tanh', 'tanh_norm']
hidden_options = [[4096, 2048], [2048, 1024], [4096, 2048, 1024], [2048, 1024, 512], [1024, 1024]]
lr_options = [1e-2, 1e-3, 1e-4, 1e-5]
dropout_options = [(0.0, 0.0), (0.2, 0.5)]
hyperparameter_grid = list(itertools.product(norm_options, hidden_options, lr_options, dropout_options))
print(f"Total no. of hyperparameter combinations: {len(hyperparameter_grid)}")

Total no. of hyperparameter combinations: 120


In [ ]:
checkpoint_file = ROOT_DIR / "hyperparam_checkpoint.json"
best_val_loss = np.inf
best_params = None
best_epoch = None
start_idx = 0
if checkpoint_file.exists():
    with open(checkpoint_file, "r") as f:
        checkpoint = json.load(f)
    best_val_loss = checkpoint.get("best_val_loss", np.inf)
    best_params = checkpoint.get("best_params", None)
    best_epoch = checkpoint.get("best_epoch", None)
    start_idx = checkpoint.get("last_completed_idx", -1) + 1
    records = checkpoint.get("records", [])
    print(f"Resuming from index {start_idx}, best_val_loss so far: {best_val_loss}")
else:
    records = []

Resuming from index 120,best_val_loss so far:0.5445041060447693


In [ ]:
data_cache = {}
for norm in norm_options:
    train_features, val_features, _, _, train_targets, val_targets, _, _ = load(norm=norm)
    data_cache[norm] = (train_features, val_features, train_targets, val_targets)
    print(f"\n{norm}")
    print("Train features shape:", train_features.shape)
    print("Val features shape:", val_features.shape)
    print("Train targets shape:", train_targets.shape)
    print("Val targets shape:", val_targets.shape)
    print("NaN in train_features:", np.isnan(train_features).any())
    print("Inf in train_features:", np.isinf(train_features).any())
    print("NaN in train_targets:", np.isnan(train_targets).any())
    print("Inf in train_targets:", np.isinf(train_targets).any())
    print("\nFirst 5 rows of train_features:\n", train_features[:5])
    print("First 5 elements of train_targets:\n", train_targets[:5])



norm
X_tr shape: (13884, 7063)
X_val shape: (4614, 7063)
y_tr shape: (13884, 1)
y_val shape: (4614, 1)
NaN in X_tr: False
Inf in X_tr: False
NaN in y_tr: False
Inf in y_tr: False

First 5 rows of X_tr:
 [[ 0.05809287 -1.2305294  -0.38594985 ...  0.          0.
   0.        ]
 [ 0.06163035 -1.2305294  -0.38594985 ... -0.561097   -0.68233347
   0.07810206]
 [-0.3140575  -1.2305294  -0.38594985 ...  0.4991565   1.9423046
  -0.38811162]
 [-0.15526268 -1.2305294  -0.38594985 ...  0.          0.
   0.        ]
 [-0.47901613 -1.2305294  -0.38594985 ...  0.          0.
   0.        ]]
First 5 elements of y_tr:
 [[ 7.69353  ]
 [ 7.7780533]
 [-1.1985054]
 [ 2.5956845]
 [-5.1399713]]

tanh
X_tr shape: (13884, 7063)
X_val shape: (4614, 7063)
y_tr shape: (13884, 1)
y_val shape: (4614, 1)
NaN in X_tr: False
Inf in X_tr: False
NaN in y_tr: False
Inf in y_tr: False

First 5 rows of X_tr:
 [[ 0.05802761 -0.84273285 -0.3678634  ...  0.          0.
   0.        ]
 [ 0.06155244 -0.84273285 -0.3678634  ..

In [13]:
def moving_average(x, n):
    return np.convolve(x, np.ones(n) / n, mode='valid')

In [ ]:
for idx, (norm_type, hidden_layers, lr, (input_dropout, hidden_dropout)) in enumerate(hyperparameter_grid):
    if idx < start_idx:
        continue

    train_features, val_features, train_targets, val_targets = data_cache[norm_type]

    K.clear_session()
    model = Sequential()
    for i, units in enumerate(hidden_layers):
        if i == 0:
            model.add(Dense(
                units, input_shape=(train_features.shape[1],), activation=ACT_FUNC, kernel_initializer='he_normal'))
            if input_dropout > 0:
                model.add(Dropout(float(input_dropout)))
        else:
            model.add(Dense(
                units, activation=ACT_FUNC, kernel_initializer='he_normal'))
            if hidden_dropout > 0:
                model.add(Dropout(float(hidden_dropout)))

    model.add(Dense(1, activation='linear', kernel_initializer='he_normal'))
    model.compile(
        loss='mean_squared_error',
        optimizer=SGD(learning_rate=float(lr), momentum=MOMENTUM)
    )
    model.summary()

    history = model.fit(
        train_features, train_targets,
        validation_data=(val_features, val_targets),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        verbose=1,
        callbacks=[tf.keras.callbacks.TerminateOnNaN()]
    )

    val_losses = np.array(history.history['val_loss'])
    local_best_epoch = int(np.argmin(val_losses))
    local_best_loss = float(val_losses[local_best_epoch])

    if local_best_loss < best_val_loss:
        best_val_loss = local_best_loss
        best_epoch = local_best_epoch + 1
        best_params = {
            "norm": norm_type,
            "hidden_layers": hidden_layers,
            "learning_rate": lr,
            "input_dropout": input_dropout,
            "hidden_dropout": hidden_dropout,
            "epochs": best_epoch
        }

    records.append({
        "local_params": {
            "norm": norm_type,
            "hidden_layers": hidden_layers,
            "learning_rate": lr,
            "input_dropout": input_dropout,
            "hidden_dropout": hidden_dropout,
        },
        "local_best_epoch": local_best_epoch,
        "local_best_loss": local_best_loss
    })

    checkpoint_data = {
        "last_completed_idx": idx,
        "best_val_loss": float(best_val_loss),
        "best_params": best_params,
        "best_epoch": best_epoch,
        "records": records
    }
    with open(checkpoint_file, "w") as f:
        json.dump(checkpoint_data, f, indent=2)

    del model
    del history
    K.clear_session()
    gc.collect()
    tf.compat.v1.reset_default_graph()

In [ ]:
out_file = ROOT_DIR / "best_hyperparams.txt"
with open(out_file, "w") as f:
    for k, v in best_params.items():
        f.write(f"{k}: {v}\n")
    f.write(f"best_val_loss: {best_val_loss}\n")